<a href="https://colab.research.google.com/github/korkutanapa/DCASE2025_TASK2/blob/main/ORJ_DCASE_EVALUATION_FILES_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!rm -rf /content/*

In [ ]:
# -*- coding: utf-8 -*-
"""
DCASE 2025 Task 2
PER-SAMPLE ADAPTIVE TDA SUBSPACE kNN
====================================

STRICT NORMAL-ONLY IDEA
-----------------------
For each unseen machine, the detector sees only:

    990 source-normal
     10 target-normal
    -----------------
    1000 normal training recordings

Each test recording is processed INDEPENDENTLY.
No test recording is used to fit, calibrate, select features for, or score
any other test recording.

For each individual test recording:

1) Look at every candidate TDA feature separately in 1-D.
2) Compare that test feature value with the corresponding NORMAL feature.
3) A feature becomes ACTIVE for that test recording when its 1-D kNN
   distance is beyond a normal leave-one-out quantile.
4) The ACTIVE features define a sample-specific subspace.
5) In exactly that subspace, calculate multivariate Euclidean kNN distance
   to the normal reference bank.
6) Re-calibrate that multivariate distance using NORMAL leave-one-out kNN
   distances calculated in THE SAME subspace.

This makes scores from, for example, a 2-D active subspace and a 20-D
active subspace comparable.

DEFAULT EXPERIMENTS
-------------------
The script generates 8 separate DCASE systems:

Candidate feature pool:
    ALL = all usable H0_/H1_ features found in train and test
    68  = fixed development-derived 68-feature pool

Normal reference strategy:
    POOLED = all 1000 normal recordings form one reference bank

    DUAL = source-normal and target-normal are modeled separately.
           A feature is ACTIVE only when it is unusual relative to BOTH.
           Final anomaly score is the MINIMUM of calibrated source and
           target multivariate anomaly ratios:
               if either normal domain explains the sample, score stays low.

Activation strictness:
    q95
    q99

Systems:
    TDA_AdaptAll_Pooled_q95
    TDA_AdaptAll_Pooled_q99
    TDA_Adapt68_Pooled_q95
    TDA_Adapt68_Pooled_q99

    TDA_AdaptAll_Dual_q95
    TDA_AdaptAll_Dual_q99
    TDA_Adapt68_Dual_q95
    TDA_Adapt68_Dual_q99

PREPROCESSING
-------------
All imputation and scaling are fitted ONLY on the 1000 normal train rows.

Each TDA feature is robustly scaled:
    center = normal median
    scale  = normal IQR
Fallback:
    MAD scale -> standard deviation -> 1

Constant features are removed.

1-D ACTIVATION
---------------
Let D_j^LOO be the normal leave-one-out 1-D kNN distance distribution
for feature j.

For a test sample x:
    r_j(x) = d_j(x) / Q_q(D_j^LOO)

Feature j is active when:
    r_j(x) > 1

Direction is saved only as a diagnostic:
    + : test scaled value >= normal scaled median
    - : test scaled value <  normal scaled median

POOLED FINAL SCORE
------------------
For the sample-specific active feature set A(x):

    d_test = mean distance to k=10 nearest points among all 1000 normals

Normal calibration in the SAME subspace:
    D_A^LOO = leave-one-out kNN distances for the 1000 normals

Final score:
    score = d_test / Q_0.95(D_A^LOO)

DUAL FINAL SCORE
----------------
For the same sample-specific subspace:

    source_ratio =
        d_source_test / Q_0.95(D_source_A^LOO)

    target_ratio =
        d_target_test / Q_0.95(D_target_A^LOO)

Final score:
    score = min(source_ratio, target_ratio)

Therefore a sample is strongly anomalous only when neither source-normal
nor target-normal explains it well.

If NO feature passes the activation threshold:
    the single most deviant feature is used as the subspace.
This does not force it to be anomalous; its final calibrated score can
remain below 1.

DECISION RESULT
---------------
A natural normal-only decision is:
    decision = 1 if final calibrated score > 1
             = 0 otherwise

This decision threshold affects only decision_result, not AUC/pAUC ranking.

OUTPUT
------
Standard DCASE files are written under:
    teams/METU/<SYSTEM_NAME>/

Diagnostics include:
    - active feature count for every test sample
    - active feature names and +/- directions
    - feature activation ratios
    - final adaptive subspace score
    - source/target component scores for DUAL systems
    - per-feature activation frequencies

ZIP:
    /content/dcase2025_adaptive_subspace_knn_submission.zip

COLAB
-----
Upload all train/test XLSX feature files and this .py to /content, then run:

    exec(open("/content/dcase2025_adaptive_subspace_knn.py").read())

The ZIP download starts automatically at the end.
"""

from __future__ import annotations

import os
import glob
import json
import math
import shutil
import warnings
import subprocess
import sys
from dataclasses import dataclass

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# =============================================================================
# 0. DEPENDENCIES
# =============================================================================

try:
    import openpyxl  # noqa: F401
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpyxl"]
    )

try:
    from sklearn.neighbors import NearestNeighbors
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "scikit-learn"]
    )
    from sklearn.neighbors import NearestNeighbors

try:
    from scipy.spatial import cKDTree
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "scipy"]
    )
    from scipy.spatial import cKDTree


# =============================================================================
# 1. CONFIGURATION
# =============================================================================

BASE_DIR = os.environ.get("TDA_BASE_DIR", "/content")

MACHINE_TYPES = [
    "AutoTrash",
    "BandSealer",
    "CoffeeGrinder",
    "HomeCamera",
    "Polisher",
    "ScrewFeeder",
    "ToyPet",
    "ToyRCCar",
]

TEAM_NAME = "METU"

EXPECTED_SOURCE_NORMAL = 990
EXPECTED_TARGET_NORMAL = 10
EXPECTED_TOTAL_NORMAL = 1000

# 1-D and final multivariate kNN.
POOLED_K = 10
SOURCE_K = 10

# Target bank has only 10 normals.
# k=9 allows train LOO calibration and test query to use the same k.
TARGET_K = 9

# Feature activation quantiles to compare.
ACTIVATION_QUANTILES = [0.95, 0.99]

# Final subspace-distance calibration is kept at q95 for every system.
FINAL_CALIBRATION_QUANTILE = 0.95

# If active set is empty, use the single most deviant feature.
MIN_SUBSPACE_FEATURES = 1

# Numerical limits.
EPS = 1e-12
MAX_RATIO = 1e6

# If True, fail when a fixed 68 feature is missing.
STRICT_68_CHECK = False

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "adaptive_subspace_knn_outputs",
)

SUBMISSION_ROOT = os.path.join(
    BASE_DIR,
    "dcase2025_adaptive_subspace_knn_submission",
)

DIAGNOSTIC_ROOT = os.path.join(
    SUBMISSION_ROOT,
    "diagnostics",
)

METADATA_ROOT = os.path.join(
    SUBMISSION_ROOT,
    "metadata",
)

ZIP_BASE = os.path.join(
    BASE_DIR,
    "dcase2025_adaptive_subspace_knn_submission",
)

ZIP_PATH = ZIP_BASE + ".zip"


# =============================================================================
# 2. FIXED DEVELOPMENT-DERIVED 68-FEATURE POOL
# =============================================================================

BEST_FEATURE_POOL_68 = [
    "H0_landscape_auc",
    "H0_landscape_l1",
    "H0_landscape_l2",
    "H0_landscape_layer5_auc",
    "H0_lifetime_skew",
    "H0_pimage_entropy",
    "H0_pimage_min",
    "H1_betti_max",
    "H1_birth_iqr",
    "H1_birth_var",
    "H1_death_iqr",
    "H1_death_max",
    "H1_death_min",
    "H1_death_var",
    "H1_min_lifetime",
    "H1_min_midlife",
    "H1_q25_lifetime",
    "H1_to_H0_num_points_ratio",
    "H1_weighted_midlife_std",

    "H0_betti_max",
    "H0_betti_num_peaks",
    "H0_landscape_layer2_max",
    "H0_landscape_layer3_max",
    "H0_mean_birth_death_ratio",
    "H0_range_lifetime",
    "H0_top3_share",
    "H1_landscape_layer2_auc",
    "H1_landscape_layer2_mean",
    "H1_landscape_layer3_mean",
    "H1_landscape_layer4_auc",
    "H1_max_lifetime",
    "H1_minus_H0_entropy",
    "H1_silhouette_std",
    "H1_std_birth_death_ratio",

    "H0_birth_skew",
    "H0_death_skew",
    "H0_persistence_entropy",
    "H0_top1_share",
    "H1_normalized_persistence_entropy",
    "H1_num_points",
    "H1_to_H0_entropy_ratio",

    "H0_death_min",
    "H0_landscape_entropy",
    "H1_death_std",
    "H1_landscape_layer2_max",
    "H1_pimage_min",

    "H0_betti_l1",
    "H0_birth_max",
    "H0_death_q25",
    "H0_landscape_layer5_max",
    "H0_max_midlife",
    "H0_median_lifetime",
    "H0_num_points",
    "H0_pimage_energy",
    "H1_birth_max",
    "H1_silhouette_entropy",
    "H1_tail_share_q90",
    "H1_tail_share_q95",

    "H0_landscape_layer1_mean",
    "H1_landscape_layer1_auc",
    "H1_to_H0_max_lifetime_ratio",

    "H0_birth_kurtosis",
    "H0_landscape_layer4_max",
    "H0_q75_lifetime",
    "H1_betti_l2",
    "H1_betti_std",
    "H1_mean_birth_death_ratio",
    "H1_persistence_entropy",
]

assert len(BEST_FEATURE_POOL_68) == 68
assert len(set(BEST_FEATURE_POOL_68)) == 68


# =============================================================================
# 3. SYSTEM DEFINITIONS
# =============================================================================

def q_name(q: float) -> str:
    return f"q{int(round(100 * q))}"


def build_systems():
    systems = []

    for pool in ["All", "68"]:
        for mode in ["Pooled", "Dual"]:
            for q in ACTIVATION_QUANTILES:
                systems.append(
                    {
                        "system_name": (
                            f"TDA_Adapt{pool}_{mode}_{q_name(q)}"
                        ),
                        "pool": pool,
                        "mode": mode,
                        "activation_q": float(q),
                    }
                )

    return systems


SYSTEMS = build_systems()


def team_system_dir(system_name: str) -> str:
    return os.path.join(
        SUBMISSION_ROOT,
        "teams",
        TEAM_NAME,
        system_name,
    )


# =============================================================================
# 4. FILE HELPERS
# =============================================================================

def latest_file(files):
    files = [
        f
        for f in sorted(set(files))
        if os.path.isfile(f)
    ]

    if not files:
        return None

    return max(
        files,
        key=os.path.getmtime,
    )


def find_train_file(machine: str):
    patterns = [
        os.path.join(
            BASE_DIR,
            f"cubical_mel_tda_features_{machine}_thr*.xlsx",
        ),
        os.path.join(
            BASE_DIR,
            "**",
            f"cubical_mel_tda_features_{machine}_thr*.xlsx",
        ),
    ]

    files = []

    for pattern in patterns:
        files.extend(
            glob.glob(
                pattern,
                recursive=True,
            )
        )

    return latest_file(
        files
    )


def find_test_file(machine: str):
    patterns = [
        os.path.join(
            BASE_DIR,
            f"cubical_mel_tda_features_{machine}*.xlsx",
        ),
        os.path.join(
            BASE_DIR,
            "**",
            f"cubical_mel_tda_features_{machine}*.xlsx",
        ),
    ]

    files = []

    for pattern in patterns:
        files.extend(
            glob.glob(
                pattern,
                recursive=True,
            )
        )

    files = [
        p
        for p in files
        if "_thr" not in os.path.basename(p).lower()
        and "_train" not in os.path.basename(p).lower()
    ]

    return latest_file(
        files
    )


def get_file_ids(df: pd.DataFrame) -> np.ndarray:
    if "file_id" in df.columns:
        return (
            df["file_id"]
            .astype(str)
            .to_numpy()
        )

    for col in [
        "file_path",
        "filename",
        "path",
        "wav_path",
    ]:
        if col in df.columns:
            return (
                df[col]
                .astype(str)
                .map(os.path.basename)
                .to_numpy()
            )

    return np.asarray(
        [
            f"sample_{i:04d}.wav"
            for i in range(len(df))
        ],
        dtype=str,
    )


# =============================================================================
# 5. NORMAL TRAIN SOURCE/TARGET DOMAIN INFERENCE
# =============================================================================

def normalize_domain_value(v) -> str:
    s = str(v).strip().lower()

    if "target" in s:
        return "target"

    if "source" in s:
        return "source"

    if s in {"t", "tgt"}:
        return "target"

    if s in {"s", "src"}:
        return "source"

    return "unknown"


def infer_train_domains(
    df: pd.DataFrame,
) -> np.ndarray:
    """
    Infer domain only in the 1000 NORMAL TRAIN rows.
    """
    domains = np.full(
        len(df),
        "unknown",
        dtype=object,
    )

    lower_to_real = {
        str(c).lower(): c
        for c in df.columns
    }

    for candidate in [
        "domain",
        "domain_label",
        "source_target",
        "source/target",
        "data_domain",
    ]:
        if candidate in lower_to_real:
            col = lower_to_real[
                candidate
            ]

            parsed = (
                df[col]
                .astype(str)
                .map(normalize_domain_value)
                .to_numpy(dtype=object)
            )

            known = (
                parsed != "unknown"
            )

            domains[
                known
            ] = parsed[
                known
            ]

    # Filename/path metadata.
    text = pd.Series(
        "",
        index=df.index,
        dtype=str,
    )

    for col in [
        "file_id",
        "file_path",
        "filename",
        "path",
        "wav_path",
    ]:
        if col in df.columns:
            text = (
                text
                + " "
                + df[col]
                .astype(str)
                .str.lower()
            )

    src_mask = text.str.contains(
        "source",
        regex=False,
        na=False,
    ).to_numpy()

    tgt_mask = text.str.contains(
        "target",
        regex=False,
        na=False,
    ).to_numpy()

    unknown = (
        domains == "unknown"
    )

    domains[
        unknown & src_mask
    ] = "source"

    unknown = (
        domains == "unknown"
    )

    domains[
        unknown & tgt_mask
    ] = "target"

    return domains


# =============================================================================
# 6. NORMAL-ONLY ROBUST PREPROCESSING
# =============================================================================

@dataclass
class PreprocessedData:
    feature_names: list[str]
    X_normal: np.ndarray
    X_test: np.ndarray
    source_mask: np.ndarray
    target_mask: np.ndarray
    centers: np.ndarray
    scales: np.ndarray


def numeric_column(
    df: pd.DataFrame,
    feature: str,
) -> np.ndarray:
    x = pd.to_numeric(
        df[feature],
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x[
        ~np.isfinite(x)
    ] = np.nan

    return x


def discover_common_tda_features(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
) -> list[str]:
    return sorted(
        [
            c
            for c in train_df.columns
            if (
                str(c).startswith("H0_")
                or str(c).startswith("H1_")
            )
            and c in test_df.columns
        ]
    )


def robust_center_and_scale(
    x: np.ndarray,
):
    finite = x[
        np.isfinite(x)
    ]

    if len(finite) == 0:
        return None

    center = float(
        np.median(
            finite
        )
    )

    q25, q75 = np.quantile(
        finite,
        [0.25, 0.75],
    )

    iqr = float(
        q75 - q25
    )

    mad = float(
        np.median(
            np.abs(
                finite - center
            )
        )
    )

    mad_scale = (
        1.4826 * mad
    )

    std = float(
        np.std(
            finite
        )
    )

    # Prefer IQR because this is explicitly robust feature-wise scaling.
    for candidate in [
        iqr,
        mad_scale,
        std,
    ]:
        if (
            np.isfinite(candidate)
            and candidate > EPS
        ):
            return (
                center,
                float(candidate),
            )

    # Constant feature -> remove.
    return None


def preprocess_machine(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    train_domains: np.ndarray,
) -> PreprocessedData:
    common = discover_common_tda_features(
        train_df,
        test_df,
    )

    source_mask = (
        train_domains == "source"
    )

    target_mask = (
        train_domains == "target"
    )

    feature_names = []
    normal_cols = []
    test_cols = []
    centers = []
    scales = []

    for feature in common:
        x_normal = numeric_column(
            train_df,
            feature,
        )

        x_test = numeric_column(
            test_df,
            feature,
        )

        cs = robust_center_and_scale(
            x_normal
        )

        if cs is None:
            continue

        center, scale = cs

        # Normal-only median imputation.
        x_normal = np.where(
            np.isfinite(
                x_normal
            ),
            x_normal,
            center,
        )

        x_test = np.where(
            np.isfinite(
                x_test
            ),
            x_test,
            center,
        )

        # Robust scaling fitted ONLY on normal train.
        x_normal_scaled = (
            x_normal - center
        ) / scale

        x_test_scaled = (
            x_test - center
        ) / scale

        if not (
            np.all(
                np.isfinite(
                    x_normal_scaled
                )
            )
            and np.all(
                np.isfinite(
                    x_test_scaled
                )
            )
        ):
            continue

        feature_names.append(
            feature
        )

        normal_cols.append(
            x_normal_scaled
        )

        test_cols.append(
            x_test_scaled
        )

        centers.append(
            center
        )

        scales.append(
            scale
        )

    if not feature_names:
        raise ValueError(
            "No usable common H0_/H1_ TDA features found."
        )

    X_normal = np.column_stack(
        normal_cols
    ).astype(float)

    X_test = np.column_stack(
        test_cols
    ).astype(float)

    return PreprocessedData(
        feature_names=feature_names,
        X_normal=X_normal,
        X_test=X_test,
        source_mask=source_mask,
        target_mask=target_mask,
        centers=np.asarray(
            centers,
            dtype=float,
        ),
        scales=np.asarray(
            scales,
            dtype=float,
        ),
    )


# =============================================================================
# 7. kNN HELPERS
# =============================================================================

def normal_distance_ratio(
    distance,
    normal_scores: np.ndarray,
    q: float,
):
    """
    Calibrate a test/query distance against a NORMAL LOO distance distribution.

    Standard case:
        ratio = distance / Q_q(normal LOO)

    Important discrete-feature edge case:
        Q_q can be exactly 0 because >=q fraction of normal points have
        identical local neighbors. In that case, ANY positive test distance
        is beyond the normal q-quantile and must therefore have ratio > 1.

        If some positive normal LOO distances exist:
            ratio = 1 + distance / max(positive normal LOO)

        If all normal LOO distances are zero:
            zero query distance -> 0
            positive query distance -> MAX_RATIO

    Returns:
        ratio, actual_quantile, zero_quantile_fallback_scale
    """
    normal_scores = np.asarray(
        normal_scores,
        dtype=float,
    )

    finite = normal_scores[
        np.isfinite(normal_scores)
    ]

    if len(finite) == 0:
        arr = np.asarray(distance, dtype=float)
        ratio = np.where(arr <= EPS, 0.0, MAX_RATIO)
        return ratio, 0.0, 0.0

    threshold = float(
        np.quantile(
            finite,
            q,
        )
    )

    arr = np.asarray(
        distance,
        dtype=float,
    )

    if (
        np.isfinite(threshold)
        and threshold > EPS
    ):
        ratio = arr / threshold
        return (
            np.clip(ratio, 0.0, MAX_RATIO),
            threshold,
            threshold,
        )

    positive = finite[
        finite > EPS
    ]

    if len(positive):
        fallback_scale = float(
            np.max(positive)
        )

        ratio = np.where(
            arr <= EPS,
            0.0,
            1.0 + arr / fallback_scale,
        )

        return (
            np.clip(ratio, 0.0, MAX_RATIO),
            threshold,
            fallback_scale,
        )

    ratio = np.where(
        arr <= EPS,
        0.0,
        MAX_RATIO,
    )

    return (
        np.clip(ratio, 0.0, MAX_RATIO),
        threshold,
        0.0,
    )


def loo_knn_mean_distance(
    X: np.ndarray,
    k: int,
) -> np.ndarray:
    """
    Leave-one-out mean kNN distance.

    cKDTree query includes self at distance 0; remove first returned neighbor.
    """
    X = np.asarray(
        X,
        dtype=float,
    )

    n = len(X)

    if n < 2:
        raise ValueError(
            "Need at least 2 normal samples."
        )

    k_eff = min(
        int(k),
        n - 1,
    )

    tree = cKDTree(
        X
    )

    d, _ = tree.query(
        X,
        k=k_eff + 1,
        workers=-1,
    )

    d = np.asarray(
        d,
        dtype=float,
    )

    if d.ndim == 1:
        d = d[
            :,
            None,
        ]

    return np.mean(
        d[
            :,
            1:k_eff + 1,
        ],
        axis=1,
    )


def query_knn_mean_distance(
    X_normal: np.ndarray,
    x_query: np.ndarray,
    k: int,
) -> float:
    X_normal = np.asarray(
        X_normal,
        dtype=float,
    )

    x_query = np.asarray(
        x_query,
        dtype=float,
    ).reshape(1, -1)

    k_eff = min(
        int(k),
        len(
            X_normal
        ),
    )

    tree = cKDTree(
        X_normal
    )

    d, _ = tree.query(
        x_query,
        k=k_eff,
        workers=-1,
    )

    d = np.asarray(
        d,
        dtype=float,
    ).reshape(-1)

    return float(
        np.mean(
            d
        )
    )


# =============================================================================
# 8. PRECOMPUTE 1-D FEATURE EVIDENCE
# =============================================================================

@dataclass
class OneDCache:
    pooled_test_distance: np.ndarray
    pooled_loo: list[np.ndarray]

    source_test_distance: np.ndarray
    source_loo: list[np.ndarray]

    target_test_distance: np.ndarray
    target_loo: list[np.ndarray]


def precompute_1d(
    data: PreprocessedData,
) -> OneDCache:
    """
    Matrices:
        test_distance shape = [n_test, n_features]
        loo is a list of one normal-score vector per feature
    """
    Xn = data.X_normal
    Xt = data.X_test

    Xs = Xn[
        data.source_mask
    ]

    Xg = Xn[
        data.target_mask
    ]

    n_test, n_features = Xt.shape

    pooled_test_distance = np.zeros(
        (
            n_test,
            n_features,
        ),
        dtype=float,
    )

    source_test_distance = np.zeros(
        (
            n_test,
            n_features,
        ),
        dtype=float,
    )

    target_test_distance = np.zeros(
        (
            n_test,
            n_features,
        ),
        dtype=float,
    )

    pooled_loo = []
    source_loo = []
    target_loo = []

    for j in range(
        n_features
    ):
        pooled_col = Xn[
            :,
            j:j+1,
        ]

        source_col = Xs[
            :,
            j:j+1,
        ]

        target_col = Xg[
            :,
            j:j+1,
        ]

        test_col = Xt[
            :,
            j:j+1,
        ]

        p_loo = loo_knn_mean_distance(
            pooled_col,
            POOLED_K,
        )

        s_loo = loo_knn_mean_distance(
            source_col,
            SOURCE_K,
        )

        t_loo = loo_knn_mean_distance(
            target_col,
            TARGET_K,
        )

        pooled_loo.append(
            p_loo
        )

        source_loo.append(
            s_loo
        )

        target_loo.append(
            t_loo
        )

        # Query all test points at once using cKDTree.
        p_tree = cKDTree(
            pooled_col
        )

        s_tree = cKDTree(
            source_col
        )

        t_tree = cKDTree(
            target_col
        )

        p_d, _ = p_tree.query(
            test_col,
            k=min(
                POOLED_K,
                len(
                    pooled_col
                ),
            ),
            workers=-1,
        )

        s_d, _ = s_tree.query(
            test_col,
            k=min(
                SOURCE_K,
                len(
                    source_col
                ),
            ),
            workers=-1,
        )

        t_d, _ = t_tree.query(
            test_col,
            k=min(
                TARGET_K,
                len(
                    target_col
                ),
            ),
            workers=-1,
        )

        pooled_test_distance[
            :,
            j,
        ] = np.mean(
            np.atleast_2d(
                p_d
            ),
            axis=1,
        )

        source_test_distance[
            :,
            j,
        ] = np.mean(
            np.atleast_2d(
                s_d
            ),
            axis=1,
        )

        target_test_distance[
            :,
            j,
        ] = np.mean(
            np.atleast_2d(
                t_d
            ),
            axis=1,
        )

        if (
            (j + 1) % 25 == 0
            or (j + 1) == n_features
        ):
            print(
                f"    1-D normal calibration {j + 1}/{n_features}"
            )

    return OneDCache(
        pooled_test_distance=pooled_test_distance,
        pooled_loo=pooled_loo,
        source_test_distance=source_test_distance,
        source_loo=source_loo,
        target_test_distance=target_test_distance,
        target_loo=target_loo,
    )


# =============================================================================
# 9. ACTIVE FEATURE RATIOS
# =============================================================================

def build_activation_ratio_matrix(
    cache: OneDCache,
    pool_indices: np.ndarray,
    mode: str,
    activation_q: float,
) -> np.ndarray:
    """
    Returns [n_test, n_pool_features].

    POOLED:
        ratio_j = calibrated 1-D distance to pooled normal bank

    DUAL:
        ratio_j = min(source calibrated ratio, target calibrated ratio)

    Thus in DUAL mode, a feature is active only when BOTH normal domains
    consider it unusual.
    """
    pool_indices = np.asarray(
        pool_indices,
        dtype=int,
    )

    n_test = cache.pooled_test_distance.shape[0]
    ratio = np.zeros(
        (n_test, len(pool_indices)),
        dtype=float,
    )

    if mode == "Pooled":
        for local_j, global_j in enumerate(pool_indices):
            r, _, _ = normal_distance_ratio(
                cache.pooled_test_distance[:, global_j],
                cache.pooled_loo[global_j],
                activation_q,
            )
            ratio[:, local_j] = r

        return np.clip(
            ratio,
            0.0,
            MAX_RATIO,
        )

    if mode == "Dual":
        for local_j, global_j in enumerate(pool_indices):
            rs, _, _ = normal_distance_ratio(
                cache.source_test_distance[:, global_j],
                cache.source_loo[global_j],
                activation_q,
            )

            rt, _, _ = normal_distance_ratio(
                cache.target_test_distance[:, global_j],
                cache.target_loo[global_j],
                activation_q,
            )

            ratio[:, local_j] = np.minimum(
                rs,
                rt,
            )

        return np.clip(
            ratio,
            0.0,
            MAX_RATIO,
        )

    raise ValueError(
        f"Unknown mode: {mode}"
    )


# =============================================================================
# 10. FINAL ADAPTIVE SUBSPACE SCORE
# =============================================================================

def pooled_subspace_score(
    X_normal: np.ndarray,
    x_test: np.ndarray,
) -> dict:
    loo = loo_knn_mean_distance(
        X_normal,
        POOLED_K,
    )

    d_test = query_knn_mean_distance(
        X_normal,
        x_test,
        POOLED_K,
    )

    ratio, q_value, fallback_scale = normal_distance_ratio(
        d_test,
        loo,
        FINAL_CALIBRATION_QUANTILE,
    )

    return {
        "score": float(np.asarray(ratio)),
        "test_distance": float(d_test),
        "normal_q95": float(q_value),
        "zero_q_fallback_scale": float(fallback_scale),
    }


def dual_subspace_score(
    X_source: np.ndarray,
    X_target: np.ndarray,
    x_test: np.ndarray,
) -> dict:
    source_loo = loo_knn_mean_distance(
        X_source,
        SOURCE_K,
    )

    target_loo = loo_knn_mean_distance(
        X_target,
        TARGET_K,
    )

    source_d = query_knn_mean_distance(
        X_source,
        x_test,
        SOURCE_K,
    )

    target_d = query_knn_mean_distance(
        X_target,
        x_test,
        TARGET_K,
    )

    source_ratio, source_q, source_fallback = normal_distance_ratio(
        source_d,
        source_loo,
        FINAL_CALIBRATION_QUANTILE,
    )

    target_ratio, target_q, target_fallback = normal_distance_ratio(
        target_d,
        target_loo,
        FINAL_CALIBRATION_QUANTILE,
    )

    source_ratio = float(np.asarray(source_ratio))
    target_ratio = float(np.asarray(target_ratio))

    score = min(
        source_ratio,
        target_ratio,
    )

    return {
        "score": float(np.clip(score, 0.0, MAX_RATIO)),
        "source_ratio": float(np.clip(source_ratio, 0.0, MAX_RATIO)),
        "target_ratio": float(np.clip(target_ratio, 0.0, MAX_RATIO)),
        "source_test_distance": float(source_d),
        "target_test_distance": float(target_d),
        "source_normal_q95": float(source_q),
        "target_normal_q95": float(target_q),
        "source_zero_q_fallback_scale": float(source_fallback),
        "target_zero_q_fallback_scale": float(target_fallback),
    }


def score_one_system(
    data: PreprocessedData,
    cache: OneDCache,
    pool_indices: np.ndarray,
    system_cfg: dict,
    file_ids: np.ndarray,
):
    mode = system_cfg[
        "mode"
    ]

    activation_q = float(
        system_cfg[
            "activation_q"
        ]
    )

    system_name = system_cfg[
        "system_name"
    ]

    pool_indices = np.asarray(
        pool_indices,
        dtype=int,
    )

    pool_names = [
        data.feature_names[j]
        for j in pool_indices
    ]

    ratio_matrix = build_activation_ratio_matrix(
        cache=cache,
        pool_indices=pool_indices,
        mode=mode,
        activation_q=activation_q,
    )

    X_normal_pool = data.X_normal[
        :,
        pool_indices,
    ]

    X_test_pool = data.X_test[
        :,
        pool_indices,
    ]

    X_source_pool = X_normal_pool[
        data.source_mask
    ]

    X_target_pool = X_normal_pool[
        data.target_mask
    ]

    n_test = len(
        X_test_pool
    )

    final_scores = np.zeros(
        n_test,
        dtype=float,
    )

    decisions = np.zeros(
        n_test,
        dtype=int,
    )

    rows = []

    active_frequency = np.zeros(
        len(
            pool_indices
        ),
        dtype=int,
    )

    # Cache final subspace normal calibrations by active-index tuple.
    pooled_calibration_cache = {}
    dual_calibration_cache = {}

    for i in range(
        n_test
    ):
        ratios = ratio_matrix[
            i
        ]

        active_local = np.flatnonzero(
            ratios > 1.0
        )

        forced_fallback = False

        if len(
            active_local
        ) < MIN_SUBSPACE_FEATURES:
            forced_fallback = True

            # Take most deviant feature(s), but do NOT force anomaly decision.
            order = np.argsort(
                -ratios,
                kind="mergesort",
            )

            active_local = order[
                :MIN_SUBSPACE_FEATURES
            ]

        active_local = np.asarray(
            active_local,
            dtype=int,
        )

        active_frequency[
            active_local
        ] += 1

        active_global = pool_indices[
            active_local
        ]

        selected_names = [
            data.feature_names[j]
            for j in active_global
        ]

        selected_ratios = ratios[
            active_local
        ]

        # +/- direction relative to NORMAL median.
        # Robust scaling centers pooled-normal median at ~0.
        test_values = data.X_test[
            i,
            active_global,
        ]

        directions = np.where(
            test_values >= 0.0,
            "+",
            "-",
        )

        active_with_direction = [
            f"{sign}{name}"
            for sign, name in zip(
                directions,
                selected_names,
            )
        ]

        key = tuple(
            int(x)
            for x in active_local.tolist()
        )

        x_sub = X_test_pool[
            i,
            active_local,
        ]

        if mode == "Pooled":
            if key not in pooled_calibration_cache:
                Xn_sub = X_normal_pool[
                    :,
                    active_local,
                ]

                loo = loo_knn_mean_distance(
                    Xn_sub,
                    POOLED_K,
                )

                pooled_calibration_cache[
                    key
                ] = {
                    "X_normal": Xn_sub,
                    "loo": loo,
                }

            cached = pooled_calibration_cache[
                key
            ]

            d_test = query_knn_mean_distance(
                cached[
                    "X_normal"
                ],
                x_sub,
                POOLED_K,
            )

            score_arr, normal_q95, zero_q_fallback_scale = normal_distance_ratio(
                d_test,
                cached[
                    "loo"
                ],
                FINAL_CALIBRATION_QUANTILE,
            )

            score = float(
                np.asarray(
                    score_arr
                )
            )

            final_scores[
                i
            ] = score

            rows.append(
                {
                    "file_id": file_ids[i],
                    "system": system_name,
                    "reference_mode": mode,
                    "activation_quantile": activation_q,
                    "candidate_pool_size": len(
                        pool_indices
                    ),
                    "n_active_features": int(
                        len(
                            active_local
                        )
                    ),
                    "forced_fallback": bool(
                        forced_fallback
                    ),
                    "active_features": ";".join(
                        selected_names
                    ),
                    "active_features_signed": ";".join(
                        active_with_direction
                    ),
                    "active_feature_ratios": ";".join(
                        [
                            f"{r:.6g}"
                            for r in selected_ratios
                        ]
                    ),
                    "max_1d_ratio": float(
                        np.max(
                            ratios
                        )
                    ),
                    "mean_active_1d_ratio": float(
                        np.mean(
                            selected_ratios
                        )
                    ),
                    "multivariate_test_distance": float(
                        d_test
                    ),
                    "multivariate_normal_q95": float(
                        normal_q95
                    ),
                    "multivariate_zero_q_fallback_scale": float(
                        zero_q_fallback_scale
                    ),
                    "source_ratio": np.nan,
                    "target_ratio": np.nan,
                    "final_anomaly_score": score,
                }
            )

        elif mode == "Dual":
            if key not in dual_calibration_cache:
                Xs_sub = X_source_pool[
                    :,
                    active_local,
                ]

                Xt_sub = X_target_pool[
                    :,
                    active_local,
                ]

                source_loo = loo_knn_mean_distance(
                    Xs_sub,
                    SOURCE_K,
                )

                target_loo = loo_knn_mean_distance(
                    Xt_sub,
                    TARGET_K,
                )

                dual_calibration_cache[
                    key
                ] = {
                    "X_source": Xs_sub,
                    "X_target": Xt_sub,
                    "source_loo": source_loo,
                    "target_loo": target_loo,
                }

            cached = dual_calibration_cache[
                key
            ]

            source_d = query_knn_mean_distance(
                cached[
                    "X_source"
                ],
                x_sub,
                SOURCE_K,
            )

            target_d = query_knn_mean_distance(
                cached[
                    "X_target"
                ],
                x_sub,
                TARGET_K,
            )

            source_ratio_arr, source_q95, source_zero_q_fallback = normal_distance_ratio(
                source_d,
                cached[
                    "source_loo"
                ],
                FINAL_CALIBRATION_QUANTILE,
            )

            target_ratio_arr, target_q95, target_zero_q_fallback = normal_distance_ratio(
                target_d,
                cached[
                    "target_loo"
                ],
                FINAL_CALIBRATION_QUANTILE,
            )

            source_ratio = float(
                np.asarray(
                    source_ratio_arr
                )
            )

            target_ratio = float(
                np.asarray(
                    target_ratio_arr
                )
            )

            score = float(
                np.clip(
                    min(
                        source_ratio,
                        target_ratio,
                    ),
                    0.0,
                    MAX_RATIO,
                )
            )

            final_scores[
                i
            ] = score

            rows.append(
                {
                    "file_id": file_ids[i],
                    "system": system_name,
                    "reference_mode": mode,
                    "activation_quantile": activation_q,
                    "candidate_pool_size": len(
                        pool_indices
                    ),
                    "n_active_features": int(
                        len(
                            active_local
                        )
                    ),
                    "forced_fallback": bool(
                        forced_fallback
                    ),
                    "active_features": ";".join(
                        selected_names
                    ),
                    "active_features_signed": ";".join(
                        active_with_direction
                    ),
                    "active_feature_ratios": ";".join(
                        [
                            f"{r:.6g}"
                            for r in selected_ratios
                        ]
                    ),
                    "max_1d_ratio": float(
                        np.max(
                            ratios
                        )
                    ),
                    "mean_active_1d_ratio": float(
                        np.mean(
                            selected_ratios
                        )
                    ),
                    "multivariate_test_distance": np.nan,
                    "multivariate_normal_q95": np.nan,
                    "source_ratio": float(
                        source_ratio
                    ),
                    "target_ratio": float(
                        target_ratio
                    ),
                    "source_test_distance": float(
                        source_d
                    ),
                    "target_test_distance": float(
                        target_d
                    ),
                    "source_normal_q95": float(
                        source_q95
                    ),
                    "target_normal_q95": float(
                        target_q95
                    ),
                    "source_zero_q_fallback_scale": float(
                        source_zero_q_fallback
                    ),
                    "target_zero_q_fallback_scale": float(
                        target_zero_q_fallback
                    ),
                    "final_anomaly_score": score,
                }
            )

        else:
            raise ValueError(
                f"Unknown mode: {mode}"
            )

        # Normal-only decision:
        # calibrated score > normal q95 equivalent ratio 1.
        decisions[
            i
        ] = int(
            final_scores[
                i
            ] > 1.0
        )

    diagnostics = pd.DataFrame(
        rows
    )

    activation_frequency_df = pd.DataFrame(
        {
            "feature": pool_names,
            "n_test_active": active_frequency,
            "active_rate": (
                active_frequency.astype(float)
                / float(
                    n_test
                )
            ),
        }
    ).sort_values(
        [
            "n_test_active",
            "feature",
        ],
        ascending=[
            False,
            True,
        ],
    )

    return {
        "scores": final_scores,
        "decisions": decisions,
        "diagnostics": diagnostics,
        "activation_frequency": activation_frequency_df,
        "ratio_matrix": ratio_matrix,
    }


# =============================================================================
# 11. SAVE DCASE OUTPUT
# =============================================================================

def save_dcase_files(
    system_name: str,
    machine: str,
    file_ids: np.ndarray,
    scores: np.ndarray,
    decisions: np.ndarray,
):
    out_dir = team_system_dir(
        system_name
    )

    os.makedirs(
        out_dir,
        exist_ok=True,
    )

    score_name = (
        f"anomaly_score_{machine}_section_00_test.csv"
    )

    decision_name = (
        f"decision_result_{machine}_section_00_test.csv"
    )

    pd.DataFrame(
        {
            0: file_ids.astype(str),
            1: np.asarray(
                scores,
                dtype=float,
            ),
        }
    ).to_csv(
        os.path.join(
            out_dir,
            score_name,
        ),
        index=False,
        header=False,
    )

    pd.DataFrame(
        {
            0: file_ids.astype(str),
            1: np.asarray(
                decisions,
                dtype=int,
            ),
        }
    ).to_csv(
        os.path.join(
            out_dir,
            decision_name,
        ),
        index=False,
        header=False,
    )


# =============================================================================
# 12. RUN ONE MACHINE
# =============================================================================

def run_machine(
    machine: str,
):
    print()
    print("=" * 120)
    print(
        f"PROCESSING {machine}"
    )
    print("=" * 120)

    train_file = find_train_file(
        machine
    )

    test_file = find_test_file(
        machine
    )

    if train_file is None:
        raise FileNotFoundError(
            f"{machine}: *_thr.xlsx normal-train file not found."
        )

    if test_file is None:
        raise FileNotFoundError(
            f"{machine}: test feature .xlsx file not found."
        )

    print(
        "  TRAIN:",
        train_file,
    )

    print(
        "  TEST :",
        test_file,
    )

    train_df = pd.read_excel(
        train_file
    )

    test_df = pd.read_excel(
        test_file
    )

    train_domains = infer_train_domains(
        train_df
    )

    n_source = int(
        np.sum(
            train_domains == "source"
        )
    )

    n_target = int(
        np.sum(
            train_domains == "target"
        )
    )

    n_unknown = int(
        np.sum(
            train_domains == "unknown"
        )
    )

    print(
        f"  normal train domains: "
        f"source={n_source}, target={n_target}, unknown={n_unknown}"
    )

    if n_unknown:
        raise ValueError(
            f"{machine}: cannot infer source/target for "
            f"{n_unknown} NORMAL TRAIN rows."
        )

    if len(
        train_df
    ) != EXPECTED_TOTAL_NORMAL:
        print(
            f"  WARNING: expected {EXPECTED_TOTAL_NORMAL} normal rows, "
            f"found {len(train_df)}."
        )

    if (
        n_source != EXPECTED_SOURCE_NORMAL
        or n_target != EXPECTED_TARGET_NORMAL
    ):
        print(
            f"  WARNING: expected source/target "
            f"{EXPECTED_SOURCE_NORMAL}/{EXPECTED_TARGET_NORMAL}; "
            f"found {n_source}/{n_target}."
        )

    # -------------------------------------------------------------------------
    # Preprocess ALL common usable TDA features once.
    # -------------------------------------------------------------------------

    data = preprocess_machine(
        train_df=train_df,
        test_df=test_df,
        train_domains=train_domains,
    )

    print(
        f"  usable TDA features after normal-only constant removal: "
        f"{len(data.feature_names)}"
    )

    # Candidate pool indices.
    all_indices = np.arange(
        len(
            data.feature_names
        ),
        dtype=int,
    )

    name_to_idx = {
        f: i
        for i, f in enumerate(
            data.feature_names
        )
    }

    missing_68 = [
        f
        for f in BEST_FEATURE_POOL_68
        if f not in name_to_idx
    ]

    available_68 = [
        f
        for f in BEST_FEATURE_POOL_68
        if f in name_to_idx
    ]

    if missing_68:
        print(
            f"  fixed-68 missing features: {len(missing_68)}"
        )

        if STRICT_68_CHECK:
            raise ValueError(
                f"{machine}: missing fixed-68 features: {missing_68}"
            )

    indices_68 = np.asarray(
        [
            name_to_idx[
                f
            ]
            for f in available_68
        ],
        dtype=int,
    )

    if len(
        indices_68
    ) == 0:
        raise ValueError(
            f"{machine}: none of fixed 68 features are available."
        )

    # -------------------------------------------------------------------------
    # Precompute 1-D normal behavior ONCE.
    # -------------------------------------------------------------------------

    print(
        "  precomputing 1-D normal kNN behavior..."
    )

    cache = precompute_1d(
        data
    )

    file_ids = get_file_ids(
        test_df
    )

    machine_diag_dir = os.path.join(
        DIAGNOSTIC_ROOT,
        machine,
    )

    os.makedirs(
        machine_diag_dir,
        exist_ok=True,
    )

    machine_summary_rows = []

    for cfg in SYSTEMS:
        pool_indices = (
            all_indices
            if cfg[
                "pool"
            ] == "All"
            else indices_68
        )

        print()
        print(
            f"  SYSTEM: {cfg['system_name']} "
            f"| pool={len(pool_indices)} "
            f"| mode={cfg['mode']} "
            f"| activation={cfg['activation_q']:.2f}"
        )

        result = score_one_system(
            data=data,
            cache=cache,
            pool_indices=pool_indices,
            system_cfg=cfg,
            file_ids=file_ids,
        )

        save_dcase_files(
            system_name=cfg[
                "system_name"
            ],
            machine=machine,
            file_ids=file_ids,
            scores=result[
                "scores"
            ],
            decisions=result[
                "decisions"
            ],
        )

        sys_diag_dir = os.path.join(
            machine_diag_dir,
            cfg[
                "system_name"
            ],
        )

        os.makedirs(
            sys_diag_dir,
            exist_ok=True,
        )

        result[
            "diagnostics"
        ].to_csv(
            os.path.join(
                sys_diag_dir,
                "test_adaptive_subspace_scores.csv",
            ),
            index=False,
        )

        result[
            "activation_frequency"
        ].to_csv(
            os.path.join(
                sys_diag_dir,
                "feature_activation_frequency.csv",
            ),
            index=False,
        )

        n_active = result[
            "diagnostics"
        ][
            "n_active_features"
        ].to_numpy(
            dtype=float
        )

        scores = result[
            "scores"
        ]

        decisions = result[
            "decisions"
        ]

        row = {
            "machine": machine,
            "system": cfg[
                "system_name"
            ],
            "candidate_pool": cfg[
                "pool"
            ],
            "reference_mode": cfg[
                "mode"
            ],
            "activation_quantile": cfg[
                "activation_q"
            ],
            "candidate_pool_size": int(
                len(
                    pool_indices
                )
            ),
            "n_test": int(
                len(
                    test_df
                )
            ),
            "active_min": float(
                np.min(
                    n_active
                )
            ),
            "active_median": float(
                np.median(
                    n_active
                )
            ),
            "active_mean": float(
                np.mean(
                    n_active
                )
            ),
            "active_max": float(
                np.max(
                    n_active
                )
            ),
            "score_min": float(
                np.min(
                    scores
                )
            ),
            "score_median": float(
                np.median(
                    scores
                )
            ),
            "score_mean": float(
                np.mean(
                    scores
                )
            ),
            "score_max": float(
                np.max(
                    scores
                )
            ),
            "n_decision_anomaly": int(
                np.sum(
                    decisions
                )
            ),
        }

        machine_summary_rows.append(
            row
        )

        print(
            "    active dims min/med/mean/max = "
            f"{row['active_min']:.0f}/"
            f"{row['active_median']:.1f}/"
            f"{row['active_mean']:.2f}/"
            f"{row['active_max']:.0f}"
        )

        print(
            "    score min/med/mean/max = "
            f"{row['score_min']:.4f}/"
            f"{row['score_median']:.4f}/"
            f"{row['score_mean']:.4f}/"
            f"{row['score_max']:.4f}"
        )

    pd.DataFrame(
        machine_summary_rows
    ).to_csv(
        os.path.join(
            machine_diag_dir,
            "machine_system_summary.csv",
        ),
        index=False,
    )

    # Feature preprocessing metadata.
    pd.DataFrame(
        {
            "feature": data.feature_names,
            "normal_center_raw": data.centers,
            "normal_scale_raw": data.scales,
            "in_fixed_68": [
                f in BEST_FEATURE_POOL_68
                for f in data.feature_names
            ],
        }
    ).to_csv(
        os.path.join(
            machine_diag_dir,
            "normal_only_feature_scaling.csv",
        ),
        index=False,
    )

    with open(
        os.path.join(
            machine_diag_dir,
            "machine_info.json",
        ),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "machine": machine,
                "train_file": train_file,
                "test_file": test_file,
                "n_normal_train": int(
                    len(
                        train_df
                    )
                ),
                "n_source_normal": n_source,
                "n_target_normal": n_target,
                "n_test": int(
                    len(
                        test_df
                    )
                ),
                "n_usable_all_features": int(
                    len(
                        data.feature_names
                    )
                ),
                "n_available_fixed68": int(
                    len(
                        indices_68
                    )
                ),
                "missing_fixed68": missing_68,
                "test_to_test_information_used": False,
                "test_labels_used": False,
                "test_domain_metadata_used": False,
            },
            f,
            indent=2,
        )

    return machine_summary_rows


# =============================================================================
# 13. GLOBAL OUTPUT
# =============================================================================

def clean_outputs():
    for path in [
        OUTPUT_DIR,
        SUBMISSION_ROOT,
    ]:
        if os.path.isdir(
            path
        ):
            shutil.rmtree(
                path
            )

    if os.path.isfile(
        ZIP_PATH
    ):
        os.remove(
            ZIP_PATH
        )

    os.makedirs(
        OUTPUT_DIR,
        exist_ok=True,
    )

    os.makedirs(
        DIAGNOSTIC_ROOT,
        exist_ok=True,
    )

    os.makedirs(
        METADATA_ROOT,
        exist_ok=True,
    )

    for cfg in SYSTEMS:
        os.makedirs(
            team_system_dir(
                cfg[
                    "system_name"
                ]
            ),
            exist_ok=True,
        )


def save_global_method_info():
    with open(
        os.path.join(
            METADATA_ROOT,
            "method_config.json",
        ),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "method": (
                    "Per-sample adaptive TDA subspace kNN "
                    "with normal-only 1-D feature activation"
                ),
                "test_labels_used": False,
                "test_domain_metadata_used": False,
                "test_to_test_information_used": False,
                "normal_train_expected": {
                    "source": EXPECTED_SOURCE_NORMAL,
                    "target": EXPECTED_TARGET_NORMAL,
                    "total": EXPECTED_TOTAL_NORMAL,
                },
                "scaling": (
                    "feature-wise normal-train median / IQR; "
                    "fallback MAD then std"
                ),
                "pooled_k": POOLED_K,
                "source_k": SOURCE_K,
                "target_k": TARGET_K,
                "activation_quantiles": ACTIVATION_QUANTILES,
                "final_calibration_quantile": (
                    FINAL_CALIBRATION_QUANTILE
                ),
                "empty_active_set_fallback": (
                    f"top {MIN_SUBSPACE_FEATURES} most deviant feature(s)"
                ),
                "systems": SYSTEMS,
                "fixed68": BEST_FEATURE_POOL_68,
                "pooled_score": (
                    "test subspace kNN distance / same-subspace "
                    "normal LOO q95"
                ),
                "dual_score": (
                    "min(source calibrated subspace ratio, "
                    "target calibrated subspace ratio)"
                ),
                "binary_decision": (
                    "final calibrated score > 1"
                ),
            },
            f,
            indent=2,
        )


def create_zip():
    return shutil.make_archive(
        ZIP_BASE,
        "zip",
        root_dir=SUBMISSION_ROOT,
    )


def auto_download_colab(
    filepath: str,
):
    try:
        from google.colab import files

        print()
        print(
            "Starting ZIP download:",
            filepath,
        )

        files.download(
            filepath
        )

    except Exception as exc:
        print()
        print(
            "Automatic Colab download could not be started."
        )

        print(
            "ZIP path:",
            filepath,
        )

        print(
            "Reason:",
            str(
                exc
            ),
        )


# =============================================================================
# 14. MAIN
# =============================================================================

def main():
    print("=" * 120)
    print("DCASE 2025 TASK 2")
    print("PER-SAMPLE ADAPTIVE TDA SUBSPACE kNN")
    print("=" * 120)

    print(
        "Normal-only model. "
        "No test labels, no test domains, no test-to-test fitting."
    )

    print()
    print(
        "Systems generated:"
    )

    for cfg in SYSTEMS:
        print(
            " ",
            cfg[
                "system_name"
            ],
        )

    clean_outputs()
    save_global_method_info()

    all_summary_rows = []
    failures = []

    for machine in MACHINE_TYPES:
        try:
            rows = run_machine(
                machine
            )

            all_summary_rows.extend(
                rows
            )

        except Exception as exc:
            print()
            print(
                f"ERROR [{machine}] "
                f"{type(exc).__name__}: {exc}"
            )

            failures.append(
                {
                    "machine": machine,
                    "error_type": type(
                        exc
                    ).__name__,
                    "error": str(
                        exc
                    ),
                }
            )

    summary_df = pd.DataFrame(
        all_summary_rows
    )

    summary_path = os.path.join(
        OUTPUT_DIR,
        "adaptive_subspace_all_systems_summary.csv",
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    # Copy summary into ZIP metadata.
    summary_df.to_csv(
        os.path.join(
            METADATA_ROOT,
            "adaptive_subspace_all_systems_summary.csv",
        ),
        index=False,
    )

    if failures:
        failures_df = pd.DataFrame(
            failures
        )

        failures_df.to_csv(
            os.path.join(
                OUTPUT_DIR,
                "failures.csv",
            ),
            index=False,
        )

        failures_df.to_csv(
            os.path.join(
                METADATA_ROOT,
                "failures.csv",
            ),
            index=False,
        )

    print()
    print("#" * 120)
    print("FINAL SUMMARY")
    print("#" * 120)

    if len(
        summary_df
    ):
        print(
            summary_df.to_string(
                index=False
            )
        )

    if failures:
        print()
        print(
            "FAILURES:"
        )

        for row in failures:
            print(
                f"  {row['machine']}: "
                f"{row['error_type']}: "
                f"{row['error']}"
            )

    if not all_summary_rows:
        raise RuntimeError(
            "No machine completed. ZIP not created."
        )

    zip_path = create_zip()

    print()
    print("=" * 120)
    print("ZIP CREATED")
    print("=" * 120)
    print(
        zip_path
    )

    print()
    print(
        "Evaluate each teams/METU/<system>/ folder separately."
    )

    auto_download_colab(
        zip_path
    )


if __name__ == "__main__":
    main()